In [13]:
import os
import cv2
import random
import numpy as np
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm
import time

In [14]:
# --- CONFIGURATION ---
# Change this to the path where you saved your Kaggle images
DATASET_PATH = r"Dataset\\asl_alphabet_train"
MAX_IMAGES_PER_CLASS = 1000
IMAGE_SIZE = (128, 128)


def square_pad(crop):
    """Pad a grayscale crop to a square with black borders (no aspect distortion)."""
    h, w = crop.shape
    s = max(h, w)
    top = (s - h) // 2; bottom = s - h - top
    left = (s - w) // 2; right = s - w - left
    return cv2.copyMakeBorder(
        crop, top, bottom, left, right, cv2.BORDER_CONSTANT, value=0
    )


def segment_and_crop(img, image_size=(128, 128), padding=10):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if mask[mask.shape[0]//2, mask.shape[1]//2] == 0:
        mask = cv2.bitwise_not(mask)

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    c = max(contours, key=cv2.contourArea)
    if cv2.contourArea(c) < 1000:
        return None

    hx, hy, hw, hh = cv2.boundingRect(c)
    H, W = gray.shape
    hx = max(0, hx - padding); hy = max(0, hy - padding)
    hw = min(W - hx, hw + padding * 2); hh = min(H - hy, hh + padding * 2)

    crop = gray[hy:hy+hh, hx:hx+hw]
    if crop.size == 0:
        return None

    crop = square_pad(crop)
    return cv2.resize(crop, image_size)


baseline_features = []  # For k-NN (segmented raw pixels)
proposed_features = []  # For HOG-SVM (segmented + HOG)
labels = []

print(f"Loading up to {MAX_IMAGES_PER_CLASS} images per class...")

# Get all class folders (A-Z, plus DEL/NOTHING/SPACE)
class_folders = [f for f in os.listdir(DATASET_PATH)
                 if os.path.isdir(os.path.join(DATASET_PATH, f))]

for class_name in tqdm(class_folders, desc="Extracting Features"):
    class_folder = os.path.join(DATASET_PATH, class_name)
    all_images = os.listdir(class_folder)

    # Randomly shuffle and slice to match feature_extraction.ipynb
    if len(all_images) > MAX_IMAGES_PER_CLASS:
        selected_images = random.sample(all_images, MAX_IMAGES_PER_CLASS)
    else:
        selected_images = all_images

    for image_name in selected_images:
        img_path = os.path.join(class_folder, image_name)
        img = cv2.imread(img_path)
        if img is None:
            continue

        # --- SHARED PREPROCESSING (segment + crop + pad) ---
        seg_img = segment_and_crop(img, IMAGE_SIZE, padding=10)
        if seg_img is None:        # skip frames where no hand was segmented
            continue

        # --- PROPOSED EXTRACTION (HOG on the segmented crop) ---
        features_hog = hog(
            seg_img,
            orientations=9,
            pixels_per_cell=(16, 16),
            cells_per_block=(2, 2),
            block_norm='L2-Hys',
            visualize=False
        )
        proposed_features.append(features_hog)

        # --- BASELINE EXTRACTION (segmented raw pixels) ---
        # Same segmented crop as the proposed model, just flattened pixels,
        # so the ONLY difference vs. proposed is the feature representation
        baseline_features.append(seg_img.flatten())

        labels.append(class_name.upper())

# Convert lists to NumPy arrays for scikit-learn
X_baseline = np.array(baseline_features)
X_proposed = np.array(proposed_features)
y = np.array(labels)

# Split the data (80% training, 20% testing)
X_base_train, X_base_test, y_train, y_test = train_test_split(X_baseline, y, test_size=0.2, random_state=42)
X_prop_train, X_prop_test, _, _ = train_test_split(X_proposed, y, test_size=0.2, random_state=42)

print(f"Extraction complete! Total images loaded: {len(y)}")
print(f"Baseline Feature Shape (Segmented Pixels): {X_baseline.shape[1]}")
print(f"Proposed Feature Shape (HOG): {X_proposed.shape[1]}")

Loading up to 1000 images per class...


Extracting Features: 100%|██████████| 29/29 [01:30<00:00,  3.13s/it]


Extraction complete! Total images loaded: 29000
Baseline Feature Shape (Segmented Pixels): 16384
Proposed Feature Shape (HOG): 1764


In [15]:
print("--- Training Baseline Model (k-NN on segmented pixels) ---")
start_time = time.time()

# Basic k-NN applied directly to flattened segmented pixels
knn_baseline = KNeighborsClassifier(n_neighbors=3)
knn_baseline.fit(X_base_train, y_train)

# Predict and Evaluate
base_predictions = knn_baseline.predict(X_base_test)
base_f1 = f1_score(y_test, base_predictions, average='weighted')

print(f"Baseline Training Time: {time.time() - start_time:.2f} seconds")
print(f"Baseline F1-Score: {base_f1:.4f}")
print("\nBaseline Classification Report:")
print(classification_report(y_test, base_predictions))

--- Training Baseline Model (k-NN on segmented pixels) ---
Baseline Training Time: 25.25 seconds
Baseline F1-Score: 0.7761

Baseline Classification Report:
              precision    recall  f1-score   support

           A       0.58      0.77      0.66       180
           B       0.60      0.79      0.68       189
           C       0.78      0.77      0.78       205
           D       0.67      0.80      0.73       191
         DEL       0.85      0.87      0.86       198
           E       0.72      0.75      0.73       198
           F       0.80      0.76      0.78       201
           G       0.80      0.82      0.81       198
           H       0.80      0.77      0.78       207
           I       0.72      0.82      0.76       212
           J       0.82      0.80      0.81       212
           K       0.72      0.75      0.73       199
           L       0.87      0.81      0.84       214
           M       0.77      0.78      0.77       201
           N       0.87      0.79

In [16]:
print("--- Training Proposed Model (HOG-SVM) ---")
start_time = time.time()

# The Proposed Pipeline (matches train_model.ipynb)
proposed_pipeline = Pipeline([
    ('scaler', StandardScaler()),        # Essential for SVM math
    ('pca', PCA(n_components=0.95)),     # Retain 95% variance
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale'))  # RBF Kernel
])

proposed_pipeline.fit(X_prop_train, y_train)

# Predict and Evaluate
prop_predictions = proposed_pipeline.predict(X_prop_test)
prop_f1 = f1_score(y_test, prop_predictions, average='weighted')

print(f"Proposed Training Time: {time.time() - start_time:.2f} seconds")
print(f"Proposed F1-Score: {prop_f1:.4f}")
print("\nProposed Classification Report:")
print(classification_report(y_test, prop_predictions))

print("-" * 30)
print(f"IMPROVEMENT: The Proposed model beat the baseline by {(prop_f1 - base_f1):.4f} in F1-Score!")

--- Training Proposed Model (HOG-SVM) ---
Proposed Training Time: 103.81 seconds
Proposed F1-Score: 0.8713

Proposed Classification Report:
              precision    recall  f1-score   support

           A       0.79      0.84      0.81       180
           B       0.81      0.88      0.85       189
           C       0.91      0.90      0.91       205
           D       0.86      0.86      0.86       191
         DEL       0.78      0.94      0.85       198
           E       0.90      0.84      0.87       198
           F       0.93      0.88      0.91       201
           G       0.93      0.85      0.89       198
           H       0.88      0.94      0.91       207
           I       0.83      0.87      0.85       212
           J       0.94      0.91      0.92       212
           K       0.85      0.85      0.85       199
           L       0.97      0.93      0.95       214
           M       0.81      0.85      0.83       201
           N       0.90      0.80      0.85      